# UA-FCCL: exact explanation gate and its checks (Colab)
Run the cells **one by one, from top to bottom**. If this Colab session was used before, first click **Runtime > Restart session**.

**Files needed on your computer:** `ua_gate.py`, `sim_gate.py` (both sent in chat), and `Shill Bidding Dataset.csv`.

What this notebook does:
1. Part A: a simulation on synthetic data that checks the gate guarantee (about 2 minutes).
2. Part B: the exact gate on the nine real datasets, including a calibration-size check.

Results are saved after every dataset, so if Colab disconnects only the unfinished dataset has to be repeated.

## Cell 1: get the repo, the clustered-conformal code and one package

In [ ]:
%cd /content
!rm -rf conformal-fuzzy-ccl class-conditional-conformal
!git clone -q https://github.com/mahibgm9676-lab/conformal-fuzzy-ccl.git
!git clone -q https://github.com/tiffanyding/class-conditional-conformal.git
%cd /content/conformal-fuzzy-ccl
!pip -q install ucimlrepo
import os
os.environ['CCL_DING_DIR'] = '/content/class-conditional-conformal'
!ls ccl_helpers.py

## Cell 2: upload `ua_gate.py` and `sim_gate.py`
Click **Choose Files** and select both files together. If your computer added `(1)` to a name, the cell fixes it.

In [ ]:
import glob, os, shutil
from google.colab import files
up = files.upload()
for base in ['ua_gate', 'sim_gate']:
    c = sorted(glob.glob(base + '*.py'), key=os.path.getmtime)
    shutil.copy(c[-1], base + '.py'); print('using', c[-1])
!ls ua_gate.py sim_gate.py

## Cell 3: Part A, simulation check of the guarantee
Prints the share of repetitions in which the true unfaithful rate among shown cases exceeded delta. For `fixed_grid` this must stay at or below 0.10 (that is epsilon). `no_bound` is the gate without any bound, shown for contrast. Takes about 2 minutes.

In [ ]:
!python sim_gate.py 200

## Cell 4: download the datasets (ignore the Shill Bidding message)

In [ ]:
!python 01_prepare_data.py
!ls data

## Cell 5: upload `Shill Bidding Dataset.csv`

In [ ]:
import shutil
from google.colab import files
up = files.upload()
for fn in up:
    shutil.move(fn, 'data/Shill Bidding Dataset.csv')
!ls data

## Cell 6: load the nine datasets

In [ ]:
import numpy as np, pandas as pd, time, os
import ccl_helpers as H
from ua_gate import run_gates
files_map = {'shill': 'Shill Bidding Dataset.csv', 'wilt': 'wilt.csv', 'htru2': 'htru2.csv', 'magic': 'magic.csv',
             'pageblocks': 'pageblocks.csv', 'drybean': 'drybean.csv', 'shuttle': 'shuttle.csv',
             'yeast': 'yeast.csv', 'ecoli': 'ecoli.csv'}
data = {n: H.load('data/' + f) for n, f in files_map.items()}
for n, (X, y) in data.items():
    print(n, X.shape, 'class sizes:', np.bincount(y).tolist())

## Cell 7: quick test on the smallest dataset (a few minutes). Send me the printed table.

In [ ]:
t0 = time.time()
X, y = data['ecoli']
df = run_gates('ecoli', X, y, range(5))
print('time', round(time.time() - t0, 1), 's')
print(df.groupby(['selector', 'delta', 'gate'])[['share', 'unfaith_among_shown', 'over_delta']].mean().round(3).to_string())

## Cell 8: helper that runs a dataset and saves the result

In [ ]:
seeds_for = {'pageblocks': range(50)}
def run_group(names):
    for n in names:
        f = f'gate2_{n}.csv'
        if os.path.exists(f):
            print(n, 'already done, skipping'); continue
        t0 = time.time()
        X, y = data[n]
        d = run_gates(n, X, y, seeds_for.get(n, range(20)), cal_fracs=(1.0, 0.5, 0.25))
        d.to_csv(f, index=False)
        print(n, 'done in', round((time.time() - t0) / 60, 1), 'min')

## Cell 9: small datasets

In [ ]:
run_group(['ecoli', 'yeast', 'wilt', 'pageblocks'])

## Cell 10: medium datasets

In [ ]:
run_group(['shill', 'drybean', 'htru2', 'magic'])

## Cell 11: Shuttle, the largest

In [ ]:
run_group(['shuttle'])

## Cell 12: summary
`over_delta` is the share of splits where the unfaithful rate among shown cases was above delta. `share` is the share of cases with an explanation shown.

In [ ]:
names = list(files_map)
G2 = pd.concat([pd.read_csv(f'gate2_{n}.csv') for n in names])
pd.set_option('display.width', 220); pd.set_option('display.max_rows', 2000)
S = G2.groupby(['cal_frac', 'selector', 'delta', 'gate'])[['share', 'unfaith_among_shown', 'over_delta']].mean().round(3)
S.to_csv('gate2_summary.csv')
print(S.to_string())

## Cell 13: download everything and send it to me

In [ ]:
!zip -q gate2_results.zip gate2_*.csv sim_gate_results.csv
from google.colab import files
files.download('gate2_results.zip')